# Exercise 1 — Conceptual Questions

### 1. Q, K, V Analogy

The Query (Q), Key (K), and Value (V) vectors can be compared to a library search system.

* **Query (Q)** represents the question or information a person is searching for in the library.
* **Key (K)** represents the labels or descriptions attached to every book in the library.
* **Value (V)** represents the actual content stored inside the books.

The system compares the Query with all Keys to determine which books are most relevant. After finding the most relevant matches, it retrieves the corresponding Values (the information inside the books). The more similar a Key is to the Query, the more attention the model gives to its Value.

### 2. The Scaling Factor

The attention formula uses the scaling factor:

$$\frac{QK^T}{\sqrt{d_k}}$$

Without scaling, the dot product values can become very large when the dimensionality $d_k$ increases.

When very large numbers are passed into the softmax function, the output probabilities become extremely peaked. This means one value becomes very close to 1 while all others become very close to 0.

This creates a problem during training because the gradients become extremely small. Small gradients slow down learning and can cause the model to train poorly or become unstable. Dividing by $\sqrt{d_k}$ keeps the values in a reasonable range and helps maintain stable gradients.

### 3. Self-Attention vs. Cross-Attention

The purpose of the cross-attention layer in the Transformer decoder is to allow the decoder to look at the encoder’s output while generating each word of the translation.

* In self-attention, the decoder only looks at previously generated words.
* In cross-attention, the decoder compares its current state (Query) with the encoder outputs (Keys and Values).

This allows the decoder to focus on the most relevant parts of the input sentence at every generation step. For example, when translating a sentence, the decoder can attend to the specific source words that are most useful for predicting the next translated word.

# Exercise 2 — Implementing Scaled Dot-Product Attention

In [1]:
import torch
import torch.nn.functional as F
import math

queries = torch.randn(1, 4, 8)
keys = torch.randn(1, 4, 8)
values = torch.randn(1, 4, 8)

scores = torch.matmul(queries, keys.transpose(-2, -1))

print("Scores shape:")
print(scores.shape)

d_k = keys.size(-1)

scaled_scores = scores / math.sqrt(d_k)

attention_weights = F.softmax(scaled_scores, dim=-1)

print("\nAttention Weights:")
print(attention_weights)

output = torch.matmul(attention_weights, values)

print("\nOutput shape:")
print(output.shape)

Scores shape:
torch.Size([1, 4, 4])

Attention Weights:
tensor([[[0.2763, 0.3238, 0.1780, 0.2219],
         [0.0580, 0.0359, 0.0339, 0.8721],
         [0.2478, 0.3705, 0.3230, 0.0586],
         [0.3668, 0.1548, 0.0521, 0.4263]]])

Output shape:
torch.Size([1, 4, 8])


## Encapsulated Function Version

In [2]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(q, k, v):
    """
    Computes scaled dot-product attention.

    Args:
        q: Queries tensor
        k: Keys tensor
        v: Values tensor

    Returns:
        output: Attention output
        attention_weights: Attention probabilities
    """

    scores = torch.matmul(q, k.transpose(-2, -1))

    d_k = k.size(-1)
    scaled_scores = scores / math.sqrt(d_k)

    attention_weights = F.softmax(scaled_scores, dim=-1)

    output = torch.matmul(attention_weights, v)

    return output, attention_weights



queries = torch.randn(1, 4, 8)
keys = torch.randn(1, 4, 8)
values = torch.randn(1, 4, 8)

output, attention_weights = scaled_dot_product_attention(
    queries,
    keys,
    values
)

print("Output shape:")
print(output.shape)

print("\nAttention Weights:")
print(attention_weights)

Output shape:
torch.Size([1, 4, 8])

Attention Weights:
tensor([[[0.2504, 0.0868, 0.1388, 0.5241],
         [0.1766, 0.7426, 0.0356, 0.0452],
         [0.3193, 0.1512, 0.0821, 0.4474],
         [0.0682, 0.0307, 0.6867, 0.2144]]])


# Exercise 3 — Challenge Problem: Masking

In [3]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Computes scaled dot-product attention with optional masking.

    Args:
        q: Queries tensor
        k: Keys tensor
        v: Values tensor
        mask: Optional mask tensor

    Returns:
        output: Attention output
        attention_weights: Attention probabilities
    """

    scores = torch.matmul(q, k.transpose(-2, -1))

    d_k = k.size(-1)
    scaled_scores = scores / math.sqrt(d_k)

    if mask is not None:

        scaled_scores = scaled_scores.masked_fill(mask, -1e9)

    attention_weights = F.softmax(scaled_scores, dim=-1)

    output = torch.matmul(attention_weights, v)

    return output, attention_weights


queries = torch.randn(1, 4, 8)
keys = torch.randn(1, 4, 8)
values = torch.randn(1, 4, 8)

mask = torch.triu(torch.ones(4, 4), diagonal=1).bool()

print("Mask:")
print(mask)

output, attention_weights = scaled_dot_product_attention(
    queries,
    keys,
    values,
    mask=mask
)

print("\nAttention Weights with Mask:")
print(attention_weights)

print("\nOutput shape:")
print(output.shape)

Mask:
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

Attention Weights with Mask:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.7830, 0.2170, 0.0000, 0.0000],
         [0.2286, 0.2634, 0.5081, 0.0000],
         [0.2822, 0.2924, 0.2992, 0.1262]]])

Output shape:
torch.Size([1, 4, 8])


## Explanation of Why Masking Works

The mask replaces certain attention scores with a very large negative number such as -1e9.

When softmax is applied:
$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$$
the exponential of a very negative number becomes almost zero:
$$e^{-1e9} \approx 0$$
As a result, masked positions receive nearly zero probability and do not influence the final attention output. This prevents the decoder from looking at future tokens during text generation.